# MORPC Insights - Distributed Energy Resources (DER)

## Overview

The Public Utilities Commission of Ohio maintains a [database](https://maps.puco.ohio.gov/arcgis/rest/services/electric/Distributed_Energy_Resources/MapServer/0/) containing the locations and attributes of distributed energy resources (DER) facilities and an associated [dashboard](https://maps.puco.ohio.gov/portal/apps/dashboards/ef2586cbf54b42cd8f5af3cf5c5da296). The dashboard provides the following notes:
  - A distributed energy resource (DER) is a source of electric power that is not directly connected to a bulk power system. DER includes both generators and energy storage technologies capable of exporting active power to the electric grid.
  - Energy Storage Capacity is reflective of standalone energy storage systems, not hybrid systems where capacity is already reported/captured in the generating units.
  - "Other" Fuel Types include Waste Gas, Biofuel, Diesel, Natural Gas/Propane, Coal, Cogeneration, and Hydro.
  
This notebook produces a tileset that includes a summary of DER facilities for the MORPC 15-county region and the counties and communities therein. This notebook is the final stage in a pipeline that fetches and standardizes the DER facility data (see [morpc-renewenergyfacilities-standardize](https://github.com/morpc/morpc-renewenergyfacilities-standardize)), and summarizes summarizes it by geography (see [morpc-renewenergyfacilities-summarize](https://github.com/morpc/morpc-renewenergyfacilities-summarize)).

The process defined herein is identified as process ID #65 in the the [Master Document List](https://morpc1.sharepoint.com/:x:/s/GISteam/EfC4j3HhohZCrSZzxJdyt5cBFEqVD7zHick8ZW0INqgCYA?e=Zc2yWF). This process is primarily intended for use in workflow ID #96.  References to identifiers in the master document list will subsequently be denoted by a number in brackets, e.g. [65].

This process is dependent on upstream processes.  See the "Prerequisites" section below.

## Prerequisites

  1. Clone the [morpc-renewenergyfacilities-summarize](https://github.com/morpc/morpc-renewenergyfacilities-summarize) repository and ensure that the paths are correct in the "Define inputs" section below.
  2. Execute the morpc-renewenergyfacilities-summarize workflow (including any upstream workflows as needed) to ensure that the outputs captured in the repository are up to date.
  4. Clone the [morpc-geos-collect](https://github.com/morpc/morpc-geos-collect) repository and ensure that the paths are correct in the "Define inputs" section below.

## Setup

### Load required libraries

In [ ]:
import pandas as pd
import os
import json
import datetime
import textwrap
import matplotlib
from matplotlib import pyplot as plt
import logging
import morpc

### Start logging

In [ ]:
morpc.logs.config_logs()
logger = logging.getLogger(__name__)

### User-specified parameters

Adjust these as needed.

In [ ]:
# Years of data to be included in output as a list of two integers (first year, last year)
YEAR_RANGE = [2000,2025]

### Static parameters

Typically these should not be adjusted.

In [ ]:
# This script may pull data from outputs of upstream workflows.  The locations of these outputs are specified by their path relative
# a GitHub root directory. This is a single directory which is presumed to contain local working copies of MORPC GitHub repositories.
# Specify the path to the directory on your system where the local working copies are stored. By default, the GitHub root directory is
# assumed to be one level up from this script.
GITHUB_ROOT = "../"

# Specify the path to the directory where the input data is stored. Sometimes the data may be sourced from this location and sometimes 
# it may be sourced from elsewhere and archived here.
INPUT_DIR = os.path.normpath("./input_data")
logger.info("Input data will be archived in directory: {}".format(INPUT_DIR))

# Specify the path to the directory where the output data is stored. Typically it is not necessary to change this, and changing it for 
# established scripts may break other scripts that depend on outputs from this one.
OUTPUT_DIR = os.path.normpath("./output_data")
logger.info("Output data will be stored in directory: {}".format(OUTPUT_DIR))

# Charts produced by the script as images and Excel files will be stored in a subdirectory
# of OUTPUT_DIR with the name specified in CHART_DIRNAME
CHART_DIRNAME = "charts"
logger.info("Charts will be stored in directory: {}".format(os.path.join(OUTPUT_DIR, CHART_DIRNAME)))

# Set EXPORT_NOTEBOOK_AS_HTML to True to automatically export the notebook in HTML format when the script finishes.  Set to False to skip the export
EXPORT_NOTEBOOK_AS_HTML = True

# Not all geographies will have tiles for facilities or generating capacity in the Insights platform.
# Create an empty dictionary which will contain lists of which geographies will be included in 
# each case.
platformIncludeLists = {}

# Define the base URL where thumbnail images will be stored.
THUMBNAIL_URL_BASE = "https://raw.githubusercontent.com/morpc-insights/renewenergy-der/refs/heads/main/output_data/charts/"

# Define the base URL where target data product (ArcGIS Dashboard) is located
DATA_PRODUCT_URL_BASE = "https://www.arcgis.com/apps/dashboards/3f2b48c930294cfda824567333f001fd"

# Define the URL of the repository where the technical details for the tileset can be
# found. For this tileset this is the GitHub repository.
TECH_DETAILS_URL = "https://github.com/morpc-insights/insights-renewenergy-der"

### Define inputs

#### Create input data directory

Create input data directory if it doesn't exist.

In [ ]:
inputDir = os.path.normpath(INPUT_DIR)
if not os.path.exists(inputDir):
    os.makedirs(inputDir)

#### Summarized DER facilities data [408]

Table containing a summary of DER facilities and output capacity by geography by year. 

In [ ]:
DER_INPUT_TABLE_RESOURCE = os.path.normpath(os.path.join(GITHUB_ROOT, "morpc-renewenergyfacilities-summarize/output_data/morpc-renewenergyfacilities-der-long.resource.yaml"))
logger.info("Resource file: {}".format(DER_INPUT_TABLE_RESOURCE))

#### Geography lookup table [375]

Lookup table providing attributes and identifiers for Central Ohio geographies.

In [ ]:
GEOS_LOOKUP_TABLE_RESOURCE = os.path.normpath(os.path.join(GITHUB_ROOT, "morpc-geos-collect/output_data/morpc-geos-lookup.resource.yaml"))
logger.info("Resource file: {}".format(GEOS_LOOKUP_TABLE_RESOURCE))

### Define outputs

#### Create output data directory

Create output data directory if it doesn't exist.

In [ ]:
outputDir = os.path.normpath(OUTPUT_DIR)
if not os.path.exists(outputDir):
    os.makedirs(outputDir)   

Create the subdirectory to contain the charts if it doesn't exist.

In [ ]:
chartDir = os.path.join(outputDir, CHART_DIRNAME)
if not os.path.exists(chartDir):
    os.makedirs(chartDir)    

#### DER facilities by geography by year [408]

Long-form table of DER facilities and capacity by geography by year intended to feed an [ArcGIS Dashboard](https://www.arcgis.com/apps/dashboards/3f2b48c930294cfda824567333f001fd).

In [ ]:
FACILITIES_TABLE_FILENAME = "renewenergy-der-long.csv"
FACILITIES_TABLE_PATH = os.path.join(outputDir, FACILITIES_TABLE_FILENAME)
FACILITIES_TABLE_SCHEMA_PATH = FACILITIES_TABLE_PATH.replace(".csv",".schema.yaml")
FACILITIES_TABLE_RESOURCE_PATH = FACILITIES_TABLE_PATH.replace(".csv",".resource.yaml")
logger.info("Data: {}".format(FACILITIES_TABLE_PATH))
logger.info("Schema: {}".format(FACILITIES_TABLE_SCHEMA_PATH))
logger.info("Resource file: {}".format(FACILITIES_TABLE_RESOURCE_PATH))

## Prepare input data

### Load geography lookup table

Load the data from the source location, creating an archival copy in the input directory defined above.  Validate the data against the resource file and the schema to ensure that it complies with the schema and has not been altered.

In [ ]:
(geosRaw, geosRawResource, geosRawSchema) = morpc.frictionless.load_data(
    GEOS_LOOKUP_TABLE_RESOURCE, 
    archiveDir=inputDir
)

Inspect the data.

In [ ]:
geosRaw.head()

Create a working copy.

In [ ]:
geos = geosRaw.copy()

### Load summarized DER facility data from upstream workflows

Load the data from the source location, creating an archival copy in the input directory defined above.  Validate the data against the resource file and the schema to ensure that it complies with the schema and has not been altered.

In [ ]:
(facilitiesRaw, facilitiesRawResource, facilitiesRawSchema) = morpc.frictionless.load_data(DER_INPUT_TABLE_RESOURCE, validate=True, archiveDir=inputDir)

Inspect the data.

In [ ]:
facilitiesRaw.head()

In [ ]:
facilitiesRaw["VALUE"].describe()

Create a working copy.

In [ ]:
facilities = facilitiesRaw.copy()

## Transform data to format required by Insights platform

### Extract data for specified range of years

Extract facilities which opened in the year range specified in the "User-specified parameters" section.

In [ ]:
facilities = facilities.loc[facilities["YEAR"].isin(range(YEAR_RANGE[0], YEAR_RANGE[1]+1))].copy()

### Standardize geography names and identifiers

Extract the geographic summary level from the GEOID as a separate column.  Create another column containing human-readable descriptions of the geography levels.

In [ ]:
facilities["SUMLEVEL"] = facilities["GEOIDFQ"].apply(lambda x:x[0:3])
facilities["GEOTYPE"] = facilities["SUMLEVEL"].map(morpc.HIERARCHY_STRING_LOOKUP)

Drop the geography names that were included in the facilities data and replace them with the standard names found in MORPC's geography lookup page. Also add the county FIPS code for the county that contains the geography.  Note that the FIPS code will be null for geographies that span multiple counties.

In [ ]:
facilities = facilities.drop(columns="NAME").merge(geos[["GEOIDFQ","COUNTYFP","NAME","MUNITYPE"]], on="GEOIDFQ", how="left")

Convert the county FIPS code to a complete county GEOID by prepending the Ohio state FIPS code ("39").

In [ ]:
facilities["COUNTYFP"] = "39" + facilities["COUNTYFP"]

Create a field for the county name and look up the county name using the county GEOID.

In [ ]:
facilities["COUNTY"] = facilities["COUNTYFP"].map(morpc.CONST_COUNTY_ID_TO_NAME)

For records representing the non-incorporated portions of townships (SUMLEVEL 070), append the word "Township" to the name followed by the name of the county that contains it in parentheses.  This is necessary because township names are sometimes reused in multiple counties. 

In [ ]:
temp = facilities.loc[facilities["MUNITYPE"] == "Township"].copy()
temp["NAME"] = temp["NAME"] + " Township (" + temp["COUNTY"] + ")"
facilities.update(temp["NAME"], overwrite=True, errors="ignore")

For records representing whole counties (SUMLEVEL 050), append the word "County" to the name.  This is necessary because sometimes incorporated places or townships have the same names as counties (e.g. Delaware City and Delaware County).

In [ ]:
temp = facilities.loc[facilities["SUMLEVEL"] == "050"].copy()
temp["NAME"] = temp["NAME"] + " County"
facilities.update(temp["NAME"], overwrite=True, errors="ignore")

### Pivot the data to semi-wide form

Extract only the fields we require.

In [ ]:
facilities = facilities.filter(items=["GEOIDFQ","NAME","GEOTYPE","YEAR","METRIC","FUEL_TYPE","VALUE"], axis="columns")

Pivot the data to wide format such that each record represents the generating capacity and number of facilities using a particular fuel type that came online in a particular geography and year.

In [ ]:
facilities = facilities.pivot(index=["GEOIDFQ","NAME","GEOTYPE","YEAR","FUEL_TYPE"], columns="METRIC", values="VALUE").reset_index()
facilities.columns.name = None

### Reformat the data to comply with the output schema

Load the schema for the output data.

In [ ]:
facilitiesSchema = morpc.frictionless.load_schema(FACILITIES_TABLE_SCHEMA_PATH)
facilitiesSchema

Rename the variables to match the schema.

In [ ]:
facilities = facilities.rename(columns={
        "Capacity":"CAPACITY",
        "Facilities":"FACILITIES"
})

Extract only the variables required by the schema.

In [ ]:
facilities = facilities.filter(items=facilitiesSchema.field_names, axis="columns")

Re-cast the variables to the types specified in the schema.

In [ ]:
facilities = morpc.frictionless.cast_field_types(facilities, facilitiesSchema)

Sort the data by geography type, then geography name, then county, then year, then fuel type.

In [ ]:
facilities = facilities.sort_values(by=["GEOTYPE","NAME","YEAR","FUEL_TYPE"])

Inspect the data.

In [ ]:
facilities.head()

## Export data

Export the data to a CSV file.

In [ ]:
facilities.to_csv(FACILITIES_TABLE_PATH, index=False)

## Create resource file for exported data

Create a [Frictionless Resource file](https://specs.frictionlessdata.io/tabular-data-resource/) for the exported data. The resource file associates the schema with the CSV file and captures key metadata about the CSV file including the filesize (bytes) and MD5 checksum (hash).  This facilities validation and integrity checking of the CSV file.  The CSV file is automatically validated after the resource file is created.

In [ ]:
facilitiesResource = morpc.frictionless.create_resource(FACILITIES_TABLE_FILENAME, 
    resourcePath=FACILITIES_TABLE_RESOURCE_PATH,
    title="MORPC Insights | Distributed Energy Resources Facilities by Year", 
    name="renewenergy_der", 
    description="Count and generation capacity of Central Ohio Distributed Energy Resources facilites which opened in each year according to data maintained by the Public Utilities Commission of Central Ohio.",
    writeResource=True,
    validate=True
)
facilitiesResource

## Generate static charts

This section will generate static charts in SVG (scalable vector graphics) and Excel format.  The SVG charts will be displayed on the tiles in the Insights platform.  Both the SVG charts and the Excel charts will be made available via GitHub for general purpose usage.  We'll create PNG-formatted charts too to facilitate efficient review when creating commentary, however these will not be stored in GitHub.

### Prepare to create charts

First delete any existing contents in the chart directory. This ensures that we are not left with any stale content in the event that no new chart is produced for a geography during this run.

In [ ]:
for f in os.scandir(chartDir):
    os.remove(f)

Load a standard color set for the chart elements.

In [ ]:
colorset = json.loads(json.dumps(morpc.CONST_COLOR_CYCLES["matplotlib"]))

### Reindex to include empty observations for all years

Create index for all years for communities in the data.

In [ ]:
facilities_all_index = []
for geoid in facilities["GEOIDFQ"].unique():
    name = facilities.loc[facilities['GEOIDFQ']==geoid, 'NAME'].to_list()[0]
    geotype = facilities.loc[facilities['GEOIDFQ']==geoid, 'GEOTYPE'].to_list()[0]
    for year in list(range(YEAR_RANGE[0], YEAR_RANGE[1]+1)):
        for fuel_type in [x for x in facilities['FUEL_TYPE'].unique()]:
            facilities_all_index.append((geoid, name, geotype, year, fuel_type))

Apply new index and correct column names.

In [ ]:
facilities_all = facilities \
    .set_index(['GEOIDFQ', 'NAME', 'GEOTYPE', 'YEAR', 'FUEL_TYPE'])\
    .reindex(pd.MultiIndex.from_tuples(facilities_all_index))\
    .reset_index()\
    .rename(
        columns = {
            'level_0': 'GEOIDFQ',
            'level_1': 'NAME',
            'level_2': 'GEOTYPE',
            'level_3': 'YEAR',
            'level_4': 'FUEL_TYPE'
        }
    )

### Create charts of number of facilities coming online by year

In [ ]:
%matplotlib agg  
# The preceeding line disables display of matplotlib charts in the notebook

# Create a list to accumulate geographies for which a thumbnail is generated
platformIncludeLists["facilities"] = []

# Iterate over each geography in data set
for geoid in facilities_all["GEOIDFQ"].unique():
    
    # Extract the data for a single geography.  If the resulting dataframe is empty, this
    # means there are no records for that geography.  In that case, skip this geography
    # and restart the loop with the next geography.
    temp = facilities_all.loc[facilities_all["GEOIDFQ"] == geoid].copy()
    if(temp.empty):
        continue
    
    # Add the geography to the list to be included in the platform
    platformIncludeLists["facilities"].append(geoid)

    # Generate chart title which includes the geography name
    geoName = temp.iloc[0]["NAME"]
    title = "Distributed Energy Resources Facilities by Year Opened - {}".format(geoName)
    
    # Define the x-axis and y-axis labels. Both axes are intuitive thanks to the chart title so
    # no labels are needed
    xlabel = None
    ylabel = None
    
    # Drop the geography name and type, which are not needed in the chart
    temp = temp.filter(items=["YEAR","FUEL_TYPE","FACILITIES"], axis="columns")
    
    # Make the variable names nicer looking
    temp = temp.rename(columns={
        "YEAR":"Open year",
        "FUEL_TYPE":"Fuel type",
        "FACILITIES":"Facilities"
    })
    
    # Pivot to wide format
    temp = temp.pivot(index="Open year", columns="Fuel type", values="Facilities")
    temp.columns.name = None
    
    ## Create and annotate the plot
    # Specify a width of 8 inches and an aspect ratio of 16:9
    PLOTWIDTH = 8
    fig,ax = plt.subplots(figsize=(PLOTWIDTH,PLOTWIDTH/16*9))

    # Create a stacked bar chart with years on the x-axis and number of facilities 
    # on the y-axis
    temp.plot.bar(ax=ax, stacked=True, color=colorset)

    # Apply the title and axis labels specified above to the chart
    ax.set_title(textwrap.fill(title, 56), fontsize=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Create the legend. Wrap the legend labels to 15 characters to keep the legend compact
    # Place the legend to the right of the chart.
    handles, labels = ax.get_legend_handles_labels()
    labels = [textwrap.fill(label, 15) for label in labels]
    legend = ax.legend(handles[::-1], labels[::-1], loc='center left', bbox_to_anchor=(1, 0.5), labelspacing=1)
    
    # Add gridlines to the chart and ensure that they are drawn beneath the bars
    ax.grid(visible=True, color="lightgrey")
    ax.set_axisbelow(True)

    # Tell matplotlib to use only integer multiples for the y-axis ticks. Then convert the resulting ticks
    # to integers, extract the unique values, and sort them. Both of these steps seem to be necessary to
    # avoid having repeated tick labels when data values are small.  Surely there must be a simpler way...
    ax.get_yaxis().set_major_locator(matplotlib.ticker.MaxNLocator(nbins="auto", steps=[1, 2, 5, 10]))
    ax.set_yticks(sorted(list(set([int(x) for x in ax.get_yticks()]))))
    
    # Format the y-axis labels as integers with comma separators
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    
    # Format the y-axis labels using engineering notation (k, M)
    #ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter())
    
    # Save the figure to disk as an SVG file
    ax.figure.savefig(os.path.join(chartDir, "facilities-{0}-{1}.svg".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), bbox_extra_artists=(legend,), bbox_inches='tight')
    ax.figure.savefig(os.path.join(chartDir, "facilities-{0}-{1}.png".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), bbox_extra_artists=(legend,), bbox_inches='tight', dpi=300)

    # Destroy the matplotlib chart in memory.
    plt.close(ax.figure)

    # Create a blank Excel document to hold the data table and chart
    writer = pd.ExcelWriter(os.path.join(chartDir, "facilities-{0}-{1}.xlsx".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), engine='xlsxwriter')
    
    # Configure the presentation of the data table, as required by morpc.data_chart_to_excel()
    dataOptions = {
        "numberFormat": {
            'Open year': "0",
            'Biofuel': "#,##0",
            'Energy Storage': "#,##0",
            'Natural Gas/Propane': "#,##0",
            'Solar': "#,##0",
            'Waste Gas': "#,##0",
            'Wind': "#,##0"
        },
        "columnWidth": 20
    }

    # Configure the presentation of the chart, as required by morpc.data_chart_to_excel()
    chartOptions = {
        "subtype":"stacked",
        "colors": colorset,
        "titles": {
            "chartTitle": textwrap.fill(title, 56),
            "xTitle": xlabel,
            "yTitle": ylabel
        },
        "seriesOptions": [{"gap":100} for x in temp.columns],
        "xAxisOptions": {
            "num_font": {"size":14},
        },
        "yAxisOptions": {
            "num_font": {"size":14},
            "num_format": "#,##0",
        },
        "legendOptions":{
            "position":"bottom",
            "font":{"size":14}
        },
        "sizeOptions":{
            "x_scale":1.5,
            "y_scale":1.5
        }
    }

    # Add the data table and chart to the Excel document
    morpc.data_chart_to_excel(temp, writer, chartType="column", dataOptions=dataOptions, chartOptions=chartOptions)

    # Close the Excel document
    writer.close()    

# Reenable display of matplotlib charts in the notebook
%matplotlib inline

### Create charts of generating capacity coming online by year

In [ ]:
%matplotlib agg  
# The preceeding line disables display of matplotlib charts in the notebook

# Create a list to accumulate geographies for which a thumbnail is generated
platformIncludeLists["capacity"] = []

# Iterate over each geography in data set
for geoid in facilities_all["GEOIDFQ"].unique():
    
    # Extract the data for a single geography.  If the resulting dataframe is empty, this
    # means there are no records for that geography.  In that case, skip this geography
    # and restart the loop with the next geography.
    temp = facilities_all.loc[facilities_all["GEOIDFQ"] == geoid].copy()
    if(temp.empty):
        continue
    
    # Add the geography to the list to be included in the platform
    platformIncludeLists["capacity"].append(geoid)

    # Generate chart title which includes the geography name
    geoName = temp.iloc[0]["NAME"]
    title = "Distributed Energy Resources Capacity by Year Opened - {}".format(geoName)
    
    # Define the x-axis and y-axis labels. The x-axis is intuitive thanks to the chart title so
    # no label is needed
    xlabel = None
    ylabel = "Kilowatts (kW)"
    
    # Drop the geography name and type, which are not needed in the chart
    temp = temp.filter(items=["YEAR","FUEL_TYPE","CAPACITY"], axis="columns")
    
    # Make the variable names nicer looking
    temp = temp.rename(columns={
        "YEAR":"Open year",
        "FUEL_TYPE":"Fuel type",
        "CAPACITY":"Capacity"
    })
    
    # Pivot to wide format
    temp = temp.pivot(index="Open year", columns="Fuel type", values="Capacity")
    temp.columns.name = None
    
    ## Create and annotate the plot
    # Specify a width of 8 inches and an aspect ratio of 16:9
    PLOTWIDTH = 8
    fig,ax = plt.subplots(figsize=(PLOTWIDTH,PLOTWIDTH/16*9))

    # Create a stacked bar chart with years on the x-axis and generating capacity 
    # on the y-axis
    temp.plot.bar(ax=ax, stacked=True, color=colorset)

    # Apply the title and axis labels specified above to the chart
    ax.set_title(textwrap.fill(title, 54), fontsize=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Create the legend. Wrap the legend labels to 15 characters to keep the legend compact
    # Place the legend to the right of the chart.
    handles, labels = ax.get_legend_handles_labels()
    labels = [textwrap.fill(label, 15) for label in labels]
    legend = ax.legend(handles[::-1], labels[::-1], loc='center left', bbox_to_anchor=(1, 0.5), labelspacing=1)
    
    # Add gridlines to the chart and ensure that they are drawn beneath the bars
    ax.grid(visible=True, color="lightgrey")
    ax.set_axisbelow(True)

    # Format the y-axis labels as integers with comma separators
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    
    # Save the figure to disk as an SVG file
    ax.figure.savefig(os.path.join(chartDir, "capacity-{0}-{1}.svg".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), bbox_extra_artists=(legend,), bbox_inches='tight')
    ax.figure.savefig(os.path.join(chartDir, "capacity-{0}-{1}.png".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), bbox_extra_artists=(legend,), bbox_inches='tight', dpi=300)

    # Destroy the matplotlib chart in memory.
    plt.close(ax.figure)

    # Create a blank Excel document to hold the data table and chart
    writer = pd.ExcelWriter(os.path.join(chartDir, "capacity-{0}-{1}.xlsx".format(geoName.replace(" ","").replace("(","").replace(")",""), geoid)), engine='xlsxwriter')
    
    # Configure the presentation of the data table, as required by morpc.data_chart_to_excel()
    dataOptions = {
        "numberFormat": {
            'Open year': "0",
            'Biofuel': "#,##0.0",
            'Energy Storage': "#,##0.0",
            'Natural Gas/Propane': "#,##0.0",
            'Solar': "#,##0.0",
            'Waste Gas': "#,##0.0",
            'Wind': "#,##0.0"
        },
        "columnWidth": 20
    }

    # Configure the presentation of the chart, as required by morpc.data_chart_to_excel()
    chartOptions = {
        "subtype":"stacked",
        "colors": colorset,
        "titles": {
            "chartTitle": textwrap.fill(title, 54),
            "xTitle": xlabel,
            "yTitle": ylabel
        },
        "seriesOptions": [{"gap":100} for x in temp.columns],
        "xAxisOptions": {
            "num_font": {"size":14},
        },
        "yAxisOptions": {
            "num_font": {"size":14},
            "num_format": '#,##0',
        },
        "legendOptions":{
            "position":"bottom",
            "font":{"size":14}
        },
        "sizeOptions":{
            "x_scale":1.5,
            "y_scale":1.5
        }
    }

    # Add the data table and chart to the Excel document
    morpc.data_chart_to_excel(temp, writer, chartType="column", dataOptions=dataOptions, chartOptions=chartOptions)

    # Close the Excel document
    writer.close()    

# Reenable display of matplotlib charts in the notebook
%matplotlib inline

## Generate Insights catalog content

The content in the Insights platform is controlled by a catalog spreadsheet. Each tile to be displayed in the platform must have a record in the catalog.  This section will create the records for the tiles that display the distributed energy resources data.  Eventually this function will be performed by a separate staging script.

First specify the column names used in the catalog.

In [ ]:
columnNames=["TileID","TilesetID","GeoType","GeoName","Category","Headline","Commentary","ThumbnailURL","Contributor","Vintage","UpdateInterval","ShareURL","DataProductURL","MoreContextURL","TechDetailsURL"]

For facilities then generating capacity, collect the metadata required by the Insights platform for only the geographies that are to be included.

In [ ]:
firstTime = True
for tileset in ["facilities","capacity"]:
    # Create a new dataframe containing only the geographies for which thumbnail images were 
    # produced in the section above.
    temp = facilities.loc[facilities["GEOIDFQ"].isin(platformIncludeLists[tileset])].copy()

    # Extract only the metadata columns of interest and flatten the data to have only one 
    # record per geography. Rename the metadata fields to match the catalog fields.
    temp = temp.filter(items=["GEOIDFQ","NAME","GEOTYPE"], axis="columns") \
        .groupby("GEOIDFQ").first() \
        .reset_index() \
        .rename(columns={"NAME":"GeoName","GEOTYPE":"GeoType"})

    # Change the GeoType values to match the schema of the catalog.
    temp["GeoType"] = temp["GeoType"].map({
        "REGION15":"Region",
        "COUNTY":"County",
        "JURIS":"Community"
    })

    # Populate some placeholder fields.
    temp["TileID"] = None
    temp["TilesetID"] = "TBD - {}".format(tileset)
    temp["Category"] = "Sustainability"
    temp["Headline"] = "TBD"
    temp["Commentary"] = "TBD"

    # Generate the URLs for the thumbnail images. These will be hosted in GitHub.
    temp["ThumbnailURL"] = \
        THUMBNAIL_URL_BASE + \
        "{}-".format(tileset) + \
        temp["GeoName"].apply(lambda x:x.replace(" ","").replace("(","").replace(")","")) + \
        "-" + \
        temp["GEOIDFQ"] + \
        ".svg"

    # Populate some other simple metadata.  Vintage in this case refers to the year that 
    # the content was published in Insights. This is to give readers an idea of how old it is.  
    # UpdateInterval gives them an idea of when to expect the next version. ShareURL is a 
    # placeholder for now.
    temp["Contributor"] = "Mid-Ohio Regional Planning Commission"
    temp["Vintage"] = str(datetime.date.today().year)
    temp["UpdateInterval"] = "annually"
    temp["ShareURL"] = None
    temp["MoreContextURL"] = None

    # Generate the data product URL.  This points to an ArcGIS Dashboard that accepts URL 
    # parameters.  GEOIDFQ is passed as a parameter to tell the app to load the data for a 
    # particular geography.
    temp["DataProductURL"] = temp["GEOIDFQ"].apply(lambda geoid:"{0}#geoid={1}".format(DATA_PRODUCT_URL_BASE, geoid))

    # Generate the URLs that point to the repository for technical details. 
    temp["TechDetailsURL"] = TECH_DETAILS_URL

    # Extract only the required columns.
    temp = temp.filter(items=columnNames, axis="columns")

    # If this is the first time through the loop, populate the catalog with the contents of
    # our temporary dataframe.  Otherwise, append the contents of the temporary dataframe to
    # the existing catalog.
    if(firstTime == True):
        catalog = temp.copy()
        firstTime = False
    else:
        catalog = pd.concat([catalog, temp], axis="index")

Inspect the listing.

In [ ]:
catalog

Save the catalog to an Excel spreadsheet.

In [ ]:
catalog.to_excel("catalog.xlsx", index=False)

It is necessary to manually add these records to the master catalog or update the records already therein.  See the following file in GitHub:

https://github.com/morpc/morpc-insights/blob/main/catalog/morpc_insights_catalog.xlsx

## Export notebook as HTML

In [ ]:
if(EXPORT_NOTEBOOK_AS_HTML):
    logger.info("Exporting notebook to HTML format")    
    morpc.notebook_to_html()
else:
    logger.info("Skipping export to HTML format")    